# Modelo de predicci?n IH - versi?n Google Colab

Este notebook adapta la versi?n local para ejecutarse en Google Colab. El c?digo de `ih_core.py` queda incluido en el notebook; s?lo necesitas subir los JSON de datos cuando la celda de carga lo pida.


In [ ]:
# Dependencias m?nimas para Colab
%pip -q install joblib pandas numpy matplotlib


In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    EN_COLAB = True
except Exception:
    EN_COLAB = False

# Si prefieres trabajar desde Drive, cambia USE_GOOGLE_DRIVE a True
# y ajusta DRIVE_PROJECT_DIR a la carpeta donde tengas los JSON.
USE_GOOGLE_DRIVE = False
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/ih_prediccion"

if EN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path(DRIVE_PROJECT_DIR)
else:
    BASE_DIR = Path("/content/ih_prediccion") if EN_COLAB else Path.cwd()

BASE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(BASE_DIR)

print(f"Directorio de trabajo: {BASE_DIR}")


In [ ]:
from pathlib import Path

IH_CORE_SOURCE = 'import pandas as pd\nimport numpy as np\nimport re\nimport json\nimport math\nfrom pathlib import Path\n\ndef cargar_json(ruta_archivo):\n\twith open(ruta_archivo, "r", encoding="utf-8") as archivo:\n\t\treturn json.load(archivo)\n\nCOLUMNAS_DIAS = [\n\t("L", "lunes_i", "lunes_f"),\n\t("M", "martes_i", "martes_f"),\n\t("Mi", "miercoles_i", "miercoles_f"),\n\t("J", "jueves_i", "jueves_f"),\n\t("V", "viernes_i", "viernes_f"),\n]\n\nCOLUMNAS_OBLIGATORIAS_DF_HIST = [\n\t"tri",\n\t"tri_num",\n\t"uea",\n\t"grupo",\n\t"eco",\n\t"lunes_i",\n\t"lunes_f",\n\t"martes_i",\n\t"martes_f",\n\t"miercoles_i",\n\t"miercoles_f",\n\t"jueves_i",\n\t"jueves_f",\n\t"viernes_i",\n\t"viernes_f",\n]\n\ndef _extraer_hora_minuto_desde_texto(texto):\n\ttexto_normalizado = str(texto).strip()\n\tif not texto_normalizado:\n\t\treturn None\n\ttexto_minusculas = texto_normalizado.lower()\n\tif texto_minusculas in {"nan", "nat", "none", "null"}:\n\t\treturn None\n\tif " " in texto_normalizado:\n\t\ttexto_normalizado = texto_normalizado.split()[-1]\n\tif "T" in texto_normalizado and ":" in texto_normalizado:\n\t\ttexto_normalizado = texto_normalizado.split("T")[-1]\n\ttexto_normalizado = texto_normalizado.replace(".", ":")\n\tcoincidencia_hh_mm = re.search(r"(\\d{1,2}):(\\d{1,2})", texto_normalizado)\n\tif coincidencia_hh_mm:\n\t\thoras = int(coincidencia_hh_mm.group(1))\n\t\tminutos = int(coincidencia_hh_mm.group(2))\n\t\tif 0 <= horas <= 23 and 0 <= minutos <= 59:\n\t\t\treturn horas, minutos\n\tdigitos = re.sub(r"\\D", "", texto_normalizado)\n\tif not digitos:\n\t\treturn None\n\tif len(digitos) <= 2:\n\t\thoras = int(digitos)\n\t\tminutos = 0\n\telif len(digitos) == 3:\n\t\thoras = int(digitos[0])\n\t\tminutos = int(digitos[1:])\n\telif len(digitos) == 4:\n\t\thoras = int(digitos[:2])\n\t\tminutos = int(digitos[2:])\n\telse:\n\t\treturn None\n\tif 0 <= horas <= 23 and 0 <= minutos <= 59:\n\t\treturn horas, minutos\n\treturn None\n\ndef normalizar_texto_hora(valor):\n\tif valor is None:\n\t\treturn None\n\tif pd.isna(valor):\n\t\treturn None\n\tpartes = _extraer_hora_minuto_desde_texto(valor)\n\tif partes is None:\n\t\treturn None\n\thoras, minutos = partes\n\treturn f"{horas:02d}:{minutos:02d}"\n\ndef hora_a_minutos(texto_hora):\n\tpartes = _extraer_hora_minuto_desde_texto(texto_hora)\n\tif partes is None:\n\t\traise ValueError(f"Hora inválida: {texto_hora!r}")\n\thoras, minutos = partes\n\treturn horas * 60 + minutos\n\ndef minutos_traslape(inicio_a, fin_a, inicio_b, fin_b):\n\tinicio = max(hora_a_minutos(inicio_a), hora_a_minutos(inicio_b))\n\tfin = min(hora_a_minutos(fin_a), hora_a_minutos(fin_b))\n\treturn max(0, fin - inicio)\n\ndef parsear_bloque_fijo_regular(cadena_bloque):\n\tpartes = str(cadena_bloque).split("-", 1)\n\tif len(partes) != 2:\n\t\traise ValueError(f"Bloque fijo regular inválido: {cadena_bloque!r}")\n\tinicio = normalizar_texto_hora(partes[0].strip())\n\tfin = normalizar_texto_hora(partes[1].strip())\n\tif not inicio or not fin:\n\t\traise ValueError(f"Bloque fijo regular inválido: {cadena_bloque!r}")\n\treturn {\n\t\tdia: [{"inicio": inicio, "fin": fin}]\n\t\tfor dia, _, _ in COLUMNAS_DIAS\n\t}\n\ndef construir_admin_windows_vacios():\n\treturn {\n\t\tdia: []\n\t\tfor dia, _, _ in COLUMNAS_DIAS\n\t}\n\ndef normalizar_admin_windows(admin_windows):\n\tadmin_windows_normalizados = construir_admin_windows_vacios()\n\tif admin_windows is None:\n\t\treturn admin_windows_normalizados\n\tfor dia, ventanas in admin_windows.items():\n\t\tif dia not in admin_windows_normalizados:\n\t\t\tcontinue\n\t\tfor ventana in ventanas:\n\t\t\tinicio = normalizar_texto_hora(ventana.get("inicio"))\n\t\t\tfin = normalizar_texto_hora(ventana.get("fin"))\n\t\t\tif inicio and fin:\n\t\t\t\tadmin_windows_normalizados[dia].append({\n\t\t\t\t\t"inicio": inicio,\n\t\t\t\t\t"fin": fin,\n\t\t\t\t})\n\treturn admin_windows_normalizados\n\ndef horario_desde_row(row):\n\tsesiones = []\n\tfor dia, columna_inicio, columna_fin in COLUMNAS_DIAS:\n\t\tinicio = normalizar_texto_hora(row[columna_inicio])\n\t\tfin = normalizar_texto_hora(row[columna_fin])\n\t\tif inicio and fin:\n\t\t\tsesiones.append({\n\t\t\t\t"dia": dia,\n\t\t\t\t"inicio": inicio,\n\t\t\t\t"fin": fin,\n\t\t\t})\n\treturn {"sesiones": sesiones}\n\ndef normalizar_horario_id_desde_df_hist(row):\n\tpartes = []\n\tfor dia, columna_inicio, columna_fin in COLUMNAS_DIAS:\n\t\tinicio = normalizar_texto_hora(row[columna_inicio])\n\t\tfin = normalizar_texto_hora(row[columna_fin])\n\t\tif inicio and fin:\n\t\t\tpartes.append(f"{dia}:{inicio}-{fin}")\n\treturn "|".join(partes)\n\ndef horario_desde_horario_id(horario_id):\n\tsesiones = []\n\thorario_id_normalizado = str(horario_id).strip()\n\tif not horario_id_normalizado:\n\t\treturn {"sesiones": sesiones}\n\tfor bloque in horario_id_normalizado.split("|"):\n\t\tbloque = bloque.strip()\n\t\tif not bloque:\n\t\t\tcontinue\n\t\tif ":" not in bloque:\n\t\t\traise ValueError(f"Bloque de horario inválido: {bloque!r}")\n\t\tdia, rango = bloque.split(":", 1)\n\t\tif "-" not in rango:\n\t\t\traise ValueError(f"Rango de horario inválido: {rango!r}")\n\t\tinicio_texto, fin_texto = rango.split("-", 1)\n\t\tinicio = normalizar_texto_hora(inicio_texto)\n\t\tfin = normalizar_texto_hora(fin_texto)\n\t\tif not inicio or not fin:\n\t\t\traise ValueError(f"Horario inválido en bloque: {bloque!r}")\n\t\tsesiones.append({\n\t\t\t"dia": dia,\n\t\t\t"inicio": inicio,\n\t\t\t"fin": fin,\n\t\t})\n\treturn {"sesiones": sesiones}\n\ndef horario_id_desde_horario(horario):\n\tpartes = []\n\tfor sesion in horario["sesiones"]:\n\t\tdia = sesion["dia"]\n\t\tinicio = normalizar_texto_hora(sesion["inicio"])\n\t\tfin = normalizar_texto_hora(sesion["fin"])\n\t\tif inicio and fin:\n\t\t\tpartes.append(f"{dia}:{inicio}-{fin}")\n\treturn "|".join(partes)\n\ndef construir_fixed_window_regulares(ecos_regulares):\n\treturn {\n\t\tstr(eco): parsear_bloque_fijo_regular(bloque_fijo)\n\t\tfor eco, bloque_fijo in ecos_regulares.items()\n\t}\n\ndef normalizar_mascara_ecos(ecos):\n\tif ecos is None:\n\t\treturn {}\n\treturn {\n\t\tstr(eco): valor\n\t\tfor eco, valor in ecos.items()\n\t}\n\ndef minutos_totales_horario(horario):\n\ttotal_minutos = 0\n\tfor sesion in horario["sesiones"]:\n\t\ttotal_minutos += hora_a_minutos(sesion["fin"]) - hora_a_minutos(sesion["inicio"])\n\treturn max(0, total_minutos)\n\ndef conjunto_dias_horario(horario):\n\treturn {sesion["dia"] for sesion in horario["sesiones"]}\n\ndef similitud_patron_dias(horario_a, horario_b):\n\tdias_a = conjunto_dias_horario(horario_a)\n\tdias_b = conjunto_dias_horario(horario_b)\n\tif not dias_a and not dias_b:\n\t\treturn 0.0\n\tunion = dias_a | dias_b\n\tif not union:\n\t\treturn 0.0\n\treturn len(dias_a & dias_b) / len(union)\n\ndef similitud_horaria_por_traslape(horario_a, horario_b):\n\ttotal_a = minutos_totales_horario(horario_a)\n\ttotal_b = minutos_totales_horario(horario_b)\n\tif total_a <= 0 or total_b <= 0:\n\t\treturn 0.0\n\ttraslape_total = 0\n\tfor sesion_a in horario_a["sesiones"]:\n\t\tfor sesion_b in horario_b["sesiones"]:\n\t\t\tif sesion_a["dia"] != sesion_b["dia"]:\n\t\t\t\tcontinue\n\t\t\ttraslape_total += minutos_traslape(\n\t\t\t\tsesion_a["inicio"],\n\t\t\t\tsesion_a["fin"],\n\t\t\t\tsesion_b["inicio"],\n\t\t\t\tsesion_b["fin"],\n\t\t\t)\n\treturn min(1.0, (2.0 * traslape_total) / (total_a + total_b))\n\ndef similitud_duracion_horaria(horario_a, horario_b):\n\ttotal_a = minutos_totales_horario(horario_a)\n\ttotal_b = minutos_totales_horario(horario_b)\n\tif total_a <= 0 or total_b <= 0:\n\t\treturn 0.0\n\treturn min(total_a, total_b) / max(total_a, total_b)\n\ndef vector_circular_tiempo_horario(horario):\n\ttotal_peso = 0.0\n\tsuma_cos = 0.0\n\tsuma_sin = 0.0\n\tfor sesion in horario["sesiones"]:\n\t\tinicio = hora_a_minutos(sesion["inicio"])\n\t\tfin = hora_a_minutos(sesion["fin"])\n\t\tduracion = max(0, fin - inicio)\n\t\tif duracion <= 0:\n\t\t\tcontinue\n\t\tpunto_medio = (inicio + fin) / 2.0\n\t\tangulo = (2.0 * math.pi * punto_medio) / 1440.0\n\t\tsuma_cos += duracion * math.cos(angulo)\n\t\tsuma_sin += duracion * math.sin(angulo)\n\t\ttotal_peso += duracion\n\tif total_peso <= 0.0:\n\t\treturn np.array([0.0, 0.0], dtype=float)\n\tvector = np.array([suma_cos / total_peso, suma_sin / total_peso], dtype=float)\n\tnorma = float(np.linalg.norm(vector))\n\tif norma <= 1e-12:\n\t\treturn np.array([0.0, 0.0], dtype=float)\n\treturn vector / norma\n\ndef similitud_tiempo_circular(horario_a, horario_b):\n\tvector_a = vector_circular_tiempo_horario(horario_a)\n\tvector_b = vector_circular_tiempo_horario(horario_b)\n\tnorma_a = float(np.linalg.norm(vector_a))\n\tnorma_b = float(np.linalg.norm(vector_b))\n\tif norma_a <= 1e-12 or norma_b <= 1e-12:\n\t\treturn 0.0\n\tcoseno = float(np.dot(vector_a, vector_b))\n\tcoseno = max(-1.0, min(1.0, coseno))\n\treturn (coseno + 1.0) / 2.0\n\nclass ModeloPrediccionIHRegular:\n\tdef __init__(\n\t\tself,\n\t\tlambda_in=1.0,\n\t\tlambda_out=1.0,\n\t\tlambda_admin_pos=1.0,\n\t\tgamma_contrato=0.5,\n\t\tgamma_admin=0.15,\n\t\talpha_out=1.0,\n\t\thalf_life_trimestres=8.0,\n\t\trho_exact=0.35,\n\t\tw_overlap=0.40,\n\t\tw_duration=0.15,\n\t\tw_days=0.20,\n\t\tw_time=0.25,\n\t):\n\t\tself.lambda_in = float(lambda_in)\n\t\tself.lambda_out = float(lambda_out)\n\t\tself.lambda_admin_pos = float(lambda_admin_pos)\n\t\tself.gamma_contrato = float(gamma_contrato)\n\t\tself.gamma_admin = float(gamma_admin)\n\t\tself.alpha_out = float(alpha_out)\n\t\tself.half_life_trimestres = float(half_life_trimestres)\n\n\t\tself.rho_exact = float(rho_exact)\n\t\tself.w_overlap = float(w_overlap)\n\t\tself.w_duration = float(w_duration)\n\t\tself.w_days = float(w_days)\n\t\tself.w_time = float(w_time)\n\n\t\tself.df_hist = None\n\t\tself.tri_num_actual = None\n\t\tself.admin_windows = None\n\n\t\tself.ecos_regulares = {}\n\t\tself.ecos_irregulares = {}\n\t\tself.ecos_vigentes_union = set()\n\n\t\tself.fixed_window_regulares = None\n\t\tself.prior_global_h = None\n\t\tself.count_out_exact = None\n\t\tself.total_out = None\n\t\tself.registros_fuera_regular = None\n\n\t\tself.cache_horarios_desde_id = {}\n\t\tself.cache_afinidad_overlap = {}\n\n\tdef peso_reciente(self, tri_num_r):\n\t\tdelta = self.tri_num_actual - float(tri_num_r)\n\t\treturn 2.0 ** (-delta / self.half_life_trimestres)\n\n\tdef obtener_horario_desde_id_cache(self, horario_id):\n\t\thorario_id_texto = str(horario_id)\n\t\tif horario_id_texto not in self.cache_horarios_desde_id:\n\t\t\tself.cache_horarios_desde_id[horario_id_texto] = horario_desde_horario_id(horario_id_texto)\n\t\treturn self.cache_horarios_desde_id[horario_id_texto]\n\n\tdef pesos_kernel_normalizados(self):\n\t\tpesos = np.array(\n\t\t\t[\n\t\t\t\tmax(0.0, self.w_overlap),\n\t\t\t\tmax(0.0, self.w_duration),\n\t\t\t\tmax(0.0, self.w_days),\n\t\t\t\tmax(0.0, self.w_time),\n\t\t\t],\n\t\t\tdtype=float,\n\t\t)\n\t\tsuma = float(pesos.sum())\n\t\tif suma <= 1e-12:\n\t\t\treturn np.array([0.25, 0.25, 0.25, 0.25], dtype=float)\n\t\treturn pesos / suma\n\n\tdef firma_kernel(self):\n\t\tpesos = self.pesos_kernel_normalizados()\n\t\treturn tuple(float(round(valor, 12)) for valor in pesos)\n\n\tdef similitud_horaria_kernelizada(self, horario_objetivo, horario_historico):\n\t\tpesos = self.pesos_kernel_normalizados()\n\t\tsimilitud_overlap = similitud_horaria_por_traslape(horario_objetivo, horario_historico)\n\t\tsimilitud_duration = similitud_duracion_horaria(horario_objetivo, horario_historico)\n\t\tsimilitud_days = similitud_patron_dias(horario_objetivo, horario_historico)\n\t\tsimilitud_time = similitud_tiempo_circular(horario_objetivo, horario_historico)\n\t\tcomponentes = np.array(\n\t\t\t[\n\t\t\t\tsimilitud_overlap,\n\t\t\t\tsimilitud_duration,\n\t\t\t\tsimilitud_days,\n\t\t\t\tsimilitud_time,\n\t\t\t],\n\t\t\tdtype=float,\n\t\t)\n\t\treturn float(np.dot(pesos, componentes))\n\n\tdef construir_prior_global_horario(self, df_hist):\n\t\tcount_global_h = {}\n\t\ttotal_global = 0.0\n\t\tfor _, row in df_hist.iterrows():\n\t\t\thorario_id = row["horario_id"]\n\t\t\tif not horario_id:\n\t\t\t\tcontinue\n\t\t\tpeso = float(row["peso_reciente"])\n\t\t\tcount_global_h[horario_id] = count_global_h.get(horario_id, 0.0) + peso\n\t\t\ttotal_global += peso\n\t\tif total_global == 0.0:\n\t\t\treturn {}\n\t\treturn {\n\t\t\thorario_id: conteo / total_global\n\t\t\tfor horario_id, conteo in count_global_h.items()\n\t\t}\n\n\tdef cobertura_contrato_regular(self, eco, horario):\n\t\teco_texto = str(eco)\n\t\tif eco_texto not in self.fixed_window_regulares:\n\t\t\treturn 0.0\n\t\tfixed_window = self.fixed_window_regulares[eco_texto]\n\t\ttotal_minutos_horario = 0\n\t\ttotal_minutos_dentro = 0\n\t\tfor sesion in horario["sesiones"]:\n\t\t\tdia = sesion["dia"]\n\t\t\tinicio_clase = sesion["inicio"]\n\t\t\tfin_clase = sesion["fin"]\n\t\t\ttotal_minutos_horario += hora_a_minutos(fin_clase) - hora_a_minutos(inicio_clase)\n\t\t\tfor ventana in fixed_window.get(dia, []):\n\t\t\t\ttotal_minutos_dentro += minutos_traslape(\n\t\t\t\t\tinicio_clase,\n\t\t\t\t\tfin_clase,\n\t\t\t\t\tventana["inicio"],\n\t\t\t\t\tventana["fin"],\n\t\t\t\t)\n\t\tif total_minutos_horario <= 0:\n\t\t\treturn 0.0\n\t\treturn total_minutos_dentro / total_minutos_horario\n\n\tdef construir_componentes_fuera_regular(self, df_hist):\n\t\tcount_out_exact = {\n\t\t\teco: {}\n\t\t\tfor eco in self.fixed_window_regulares.keys()\n\t\t}\n\t\ttotal_out = {\n\t\t\teco: 0.0\n\t\t\tfor eco in self.fixed_window_regulares.keys()\n\t\t}\n\t\tregistros_fuera_regular = {\n\t\t\teco: []\n\t\t\tfor eco in self.fixed_window_regulares.keys()\n\t\t}\n\t\tfor _, row in df_hist.iterrows():\n\t\t\teco = str(row["eco"])\n\t\t\tif eco not in self.fixed_window_regulares:\n\t\t\t\tcontinue\n\t\t\thorario_id = row["horario_id"]\n\t\t\tif not horario_id:\n\t\t\t\tcontinue\n\t\t\thorario = row["horario"]\n\t\t\tafinidad_contrato = self.cobertura_contrato_regular(eco, horario)\n\t\t\tif afinidad_contrato >= 1.0:\n\t\t\t\tcontinue\n\t\t\tpeso = float(row["peso_reciente"])\n\t\t\tcount_out_exact[eco][horario_id] = count_out_exact[eco].get(horario_id, 0.0) + peso\n\t\t\ttotal_out[eco] += peso\n\t\t\tregistros_fuera_regular[eco].append(\n\t\t\t\t{\n\t\t\t\t\t"horario_id": horario_id,\n\t\t\t\t\t"horario": horario,\n\t\t\t\t\t"peso": peso,\n\t\t\t\t}\n\t\t\t)\n\t\treturn count_out_exact, total_out, registros_fuera_regular\n\n\tdef afinidad_fuera_exacta_regular(self, eco, horario_id):\n\t\teco_texto = str(eco)\n\t\treturn (\n\t\t\tself.count_out_exact.get(eco_texto, {}).get(horario_id, 0.0)\n\t\t\t+ self.alpha_out * self.prior_global_h.get(horario_id, 0.0)\n\t\t) / (\n\t\t\tself.total_out.get(eco_texto, 0.0) + self.alpha_out\n\t\t)\n\n\tdef afinidad_fuera_overlap_regular(self, eco, horario_id):\n\t\teco_texto = str(eco)\n\t\tfirma_kernel = self.firma_kernel()\n\t\tcache_key = (eco_texto, str(horario_id), firma_kernel)\n\t\tif cache_key in self.cache_afinidad_overlap:\n\t\t\treturn self.cache_afinidad_overlap[cache_key]\n\n\t\thorario_objetivo = self.obtener_horario_desde_id_cache(horario_id)\n\t\tregistros = self.registros_fuera_regular.get(eco_texto, [])\n\t\ttotal_peso = self.total_out.get(eco_texto, 0.0)\n\n\t\tif total_peso <= 0.0 or not registros:\n\t\t\tself.cache_afinidad_overlap[cache_key] = 0.0\n\t\t\treturn 0.0\n\n\t\tmasa_overlap = 0.0\n\t\tfor registro in registros:\n\t\t\tsimilitud = self.similitud_horaria_kernelizada(horario_objetivo, registro["horario"])\n\t\t\tmasa_overlap += float(registro["peso"]) * similitud\n\n\t\tafinidad_overlap = masa_overlap / total_peso\n\t\tself.cache_afinidad_overlap[cache_key] = afinidad_overlap\n\t\treturn afinidad_overlap\n\n\tdef componentes_afinidad_fuera_regular(self, eco, horario_id):\n\t\tafinidad_exacta = self.afinidad_fuera_exacta_regular(eco, horario_id)\n\t\tafinidad_overlap = self.afinidad_fuera_overlap_regular(eco, horario_id)\n\t\trho_exact_clamp = max(0.0, min(1.0, self.rho_exact))\n\t\tafinidad_total = rho_exact_clamp * afinidad_exacta + (1.0 - rho_exact_clamp) * afinidad_overlap\n\t\treturn {\n\t\t\t"afinidad_fuera_historica_exacta": afinidad_exacta,\n\t\t\t"afinidad_fuera_historica_overlap": afinidad_overlap,\n\t\t\t"afinidad_fuera_historica": afinidad_total,\n\t\t}\n\n\tdef afinidad_fuera_historica_regular(self, eco, horario_id):\n\t\treturn self.componentes_afinidad_fuera_regular(eco, horario_id)["afinidad_fuera_historica"]\n\n\tdef preferencia_admin(self, horario):\n\t\ttotal_minutos_horario = 0\n\t\ttotal_minutos_preferidos = 0\n\t\tfor sesion in horario["sesiones"]:\n\t\t\tdia = sesion["dia"]\n\t\t\tinicio_clase = sesion["inicio"]\n\t\t\tfin_clase = sesion["fin"]\n\t\t\ttotal_minutos_horario += hora_a_minutos(fin_clase) - hora_a_minutos(inicio_clase)\n\t\t\tfor ventana in self.admin_windows.get(dia, []):\n\t\t\t\ttotal_minutos_preferidos += minutos_traslape(\n\t\t\t\t\tinicio_clase,\n\t\t\t\t\tfin_clase,\n\t\t\t\t\tventana["inicio"],\n\t\t\t\t\tventana["fin"],\n\t\t\t\t)\n\t\tif total_minutos_horario <= 0:\n\t\t\treturn 0.0\n\t\treturn total_minutos_preferidos / total_minutos_horario\n\n\tdef score_horario_regular(self, eco, horario_id, horario):\n\t\tafinidad_contrato = self.cobertura_contrato_regular(eco, horario)\n\t\tcomponentes_afinidad = self.componentes_afinidad_fuera_regular(eco, horario_id)\n\t\tafinidad_fuera_historica = componentes_afinidad["afinidad_fuera_historica"]\n\t\tpreferencia_administrativa = self.preferencia_admin(horario)\n\n\t\tbonus_contrato = self.lambda_in * afinidad_contrato\n\t\tbonus_fuera_justificado = self.lambda_out * (1.0 - afinidad_contrato) * afinidad_fuera_historica\n\t\tbonus_admin = self.lambda_admin_pos * preferencia_administrativa\n\n\t\tpenalizacion_fuera_no_justificada = (\n\t\t\tself.gamma_contrato\n\t\t\t* (1.0 - afinidad_contrato)\n\t\t\t* (1.0 - afinidad_fuera_historica)\n\t\t)\n\n\t\tpenalizacion_admin = self.gamma_admin * (1.0 - preferencia_administrativa)\n\n\t\tscore_lineal = (\n\t\t\tbonus_contrato\n\t\t\t+ bonus_fuera_justificado\n\t\t\t+ bonus_admin\n\t\t\t- penalizacion_fuera_no_justificada\n\t\t\t- penalizacion_admin\n\t\t)\n\n\t\tscore_horario_regular = min(1.0, max(0.0, score_lineal))\n\n\t\treturn {\n\t\t\t"eco": str(eco),\n\t\t\t"horario_id": horario_id,\n\t\t\t"afinidad_contrato": afinidad_contrato,\n\t\t\t"afinidad_fuera_historica_exacta": componentes_afinidad["afinidad_fuera_historica_exacta"],\n\t\t\t"afinidad_fuera_historica_overlap": componentes_afinidad["afinidad_fuera_historica_overlap"],\n\t\t\t"afinidad_fuera_historica": afinidad_fuera_historica,\n\t\t\t"rho_exact": max(0.0, min(1.0, self.rho_exact)),\n\t\t\t"prior_global_h": self.prior_global_h.get(horario_id, 0.0),\n\t\t\t"preferencia_administrativa": preferencia_administrativa,\n\t\t\t"bonus_contrato": bonus_contrato,\n\t\t\t"bonus_fuera_justificado": bonus_fuera_justificado,\n\t\t\t"bonus_admin": bonus_admin,\n\t\t\t"penalizacion_fuera_no_justificada": penalizacion_fuera_no_justificada,\n\t\t\t"penalizacion_admin": penalizacion_admin,\n\t\t\t"score_lineal": score_lineal,\n\t\t\t"score_horario_regular": score_horario_regular,\n\t\t\t"kernel_weights": {\n\t\t\t\t"w_overlap": float(self.pesos_kernel_normalizados()[0]),\n\t\t\t\t"w_duration": float(self.pesos_kernel_normalizados()[1]),\n\t\t\t\t"w_days": float(self.pesos_kernel_normalizados()[2]),\n\t\t\t\t"w_time": float(self.pesos_kernel_normalizados()[3]),\n\t\t\t},\n\t\t}\n\n\tdef fit(\n\t\tself,\n\t\tdf_hist,\n\t\tecos_regulares,\n\t\tecos_irregulares=None,\n\t\tadmin_windows=None,\n\t\ttri_num_actual=None,\n\t):\n\t\tcolumnas_faltantes = [\n\t\t\tcolumna\n\t\t\tfor columna in COLUMNAS_OBLIGATORIAS_DF_HIST\n\t\t\tif columna not in df_hist.columns\n\t\t]\n\t\tif columnas_faltantes:\n\t\t\traise ValueError(f"Faltan columnas obligatorias en df_hist: {columnas_faltantes}")\n\n\t\tdf_hist_entrenamiento = df_hist.copy()\n\t\tdf_hist_entrenamiento["eco"] = df_hist_entrenamiento["eco"].astype(str)\n\t\tdf_hist_entrenamiento["tri_num"] = pd.to_numeric(df_hist_entrenamiento["tri_num"], errors="raise")\n\n\t\tfor _, columna_inicio, columna_fin in COLUMNAS_DIAS:\n\t\t\tdf_hist_entrenamiento[columna_inicio] = df_hist_entrenamiento[columna_inicio].apply(normalizar_texto_hora)\n\t\t\tdf_hist_entrenamiento[columna_fin] = df_hist_entrenamiento[columna_fin].apply(normalizar_texto_hora)\n\n\t\tdf_hist_entrenamiento["horario"] = df_hist_entrenamiento.apply(horario_desde_row, axis=1)\n\t\tdf_hist_entrenamiento["horario_id"] = df_hist_entrenamiento.apply(normalizar_horario_id_desde_df_hist, axis=1)\n\n\t\tself.df_hist = df_hist_entrenamiento\n\t\tself.tri_num_actual = float(\n\t\t\tdf_hist_entrenamiento["tri_num"].max()\n\t\t\tif tri_num_actual is None\n\t\t\telse tri_num_actual\n\t\t)\n\n\t\tself.df_hist["peso_reciente"] = self.df_hist["tri_num"].apply(self.peso_reciente)\n\n\t\tself.ecos_regulares = normalizar_mascara_ecos(ecos_regulares)\n\t\tself.ecos_irregulares = normalizar_mascara_ecos(ecos_irregulares)\n\t\tself.ecos_vigentes_union = set(self.ecos_regulares.keys()) | set(self.ecos_irregulares.keys())\n\n\t\tself.fixed_window_regulares = construir_fixed_window_regulares(self.ecos_regulares)\n\t\tself.admin_windows = normalizar_admin_windows(admin_windows)\n\n\t\tself.cache_horarios_desde_id = {}\n\t\tself.cache_afinidad_overlap = {}\n\n\t\tself.prior_global_h = self.construir_prior_global_horario(self.df_hist)\n\t\tself.count_out_exact, self.total_out, self.registros_fuera_regular = self.construir_componentes_fuera_regular(self.df_hist)\n\n\t\treturn self\n\n\tdef predecir(self, eco, horario_id=None, horario=None):\n\t\teco_texto = str(eco)\n\n\t\tif eco_texto not in self.fixed_window_regulares:\n\t\t\traise ValueError(f"El eco {eco_texto!r} no está en la máscara de regulares")\n\n\t\tif horario is None and horario_id is None:\n\t\t\traise ValueError("Debes proporcionar horario o horario_id")\n\n\t\tif isinstance(horario, str):\n\t\t\thorario_id = horario\n\t\t\thorario = None\n\n\t\tif horario is None:\n\t\t\thorario = self.obtener_horario_desde_id_cache(horario_id)\n\n\t\tif horario_id is None:\n\t\t\thorario_id = horario_id_desde_horario(horario)\n\n\t\treturn self.score_horario_regular(\n\t\t\teco=eco_texto,\n\t\t\thorario_id=horario_id,\n\t\t\thorario=horario,\n\t\t)\n\ndef parsear_bloques_inferidos(lista_inferidos):\n\tventanas_por_dia = {dia: [] for dia, _, _ in COLUMNAS_DIAS}\n\tfor bloque in lista_inferidos:\n\t\tdia_raw = bloque.get("dia")\n\t\tinicio = normalizar_texto_hora(bloque.get("inicio"))\n\t\tfin = normalizar_texto_hora(bloque.get("fin"))\n\t\tif not inicio or not fin:\n\t\t\tcontinue\n\t\tif dia_raw == "L-V":\n\t\t\tfor dia in ventanas_por_dia.keys():\n\t\t\t\tventanas_por_dia[dia].append({"inicio": inicio, "fin": fin})\n\t\telse:\n\t\t\tif dia_raw in ventanas_por_dia:\n\t\t\t\tventanas_por_dia[dia_raw].append({"inicio": inicio, "fin": fin})\n\treturn ventanas_por_dia\n\ndef construir_fixed_window_irregulares(ecos_irregulares_inferidos):\n\treturn {\n\t\tstr(eco): parsear_bloques_inferidos(data.get("inferido", []))\n\t\tfor eco, data in ecos_irregulares_inferidos.items()\n\t}\n\nclass ModeloPrediccionIHIrregular(ModeloPrediccionIHRegular):\n\tdef fit(\n\t\tself,\n\t\tdf_hist,\n\t\tecos_irregulares_inferidos,\n\t\tadmin_windows=None,\n\t\ttri_num_actual=None,\n\t):\n\t\tcolumnas_faltantes = [\n\t\t\tcolumna\n\t\t\tfor columna in COLUMNAS_OBLIGATORIAS_DF_HIST\n\t\t\tif columna not in df_hist.columns\n\t\t]\n\t\tif columnas_faltantes:\n\t\t\traise ValueError(f"Faltan columnas obligatorias en df_hist: {columnas_faltantes}")\n\n\t\tdf_hist_entrenamiento = df_hist.copy()\n\t\tdf_hist_entrenamiento["eco"] = df_hist_entrenamiento["eco"].astype(str)\n\t\tdf_hist_entrenamiento["tri_num"] = pd.to_numeric(df_hist_entrenamiento["tri_num"], errors="raise")\n\n\t\tfor _, columna_inicio, columna_fin in COLUMNAS_DIAS:\n\t\t\tdf_hist_entrenamiento[columna_inicio] = df_hist_entrenamiento[columna_inicio].apply(normalizar_texto_hora)\n\t\t\tdf_hist_entrenamiento[columna_fin] = df_hist_entrenamiento[columna_fin].apply(normalizar_texto_hora)\n\n\t\tdf_hist_entrenamiento["horario"] = df_hist_entrenamiento.apply(horario_desde_row, axis=1)\n\t\tdf_hist_entrenamiento["horario_id"] = df_hist_entrenamiento.apply(normalizar_horario_id_desde_df_hist, axis=1)\n\n\t\tself.df_hist = df_hist_entrenamiento\n\t\tself.tri_num_actual = float(\n\t\t\tdf_hist_entrenamiento["tri_num"].max()\n\t\t\tif tri_num_actual is None\n\t\t\telse tri_num_actual\n\t\t)\n\n\t\tself.df_hist["peso_reciente"] = self.df_hist["tri_num"].apply(self.peso_reciente)\n\n\t\tself.ecos_irregulares_inferidos = normalizar_mascara_ecos(ecos_irregulares_inferidos)\n\t\tself.fixed_window_irregulares = construir_fixed_window_irregulares(self.ecos_irregulares_inferidos)\n\t\t\n\t\t# Reutilizar lógica de la clase padre\n\t\tself.fixed_window_regulares = self.fixed_window_irregulares\n\t\tself.ecos_regulares = {eco: "" for eco in self.fixed_window_irregulares.keys()}\n\n\t\tself.admin_windows = normalizar_admin_windows(admin_windows)\n\n\t\tself.cache_horarios_desde_id = {}\n\t\tself.cache_afinidad_overlap = {}\n\n\t\tself.prior_global_h = self.construir_prior_global_horario(self.df_hist)\n\t\tself.count_out_exact, self.total_out, self.registros_fuera_regular = self.construir_componentes_fuera_regular(self.df_hist)\n\n\t\treturn self\n\n\tdef predecir(self, eco, horario_id=None, horario=None):\n\t\teco_texto = str(eco)\n\n\t\tif eco_texto not in self.fixed_window_irregulares:\n\t\t\traise ValueError(f"El eco {eco_texto!r} no está en la máscara de irregulares inferidos")\n\n\t\tif horario is None and horario_id is None:\n\t\t\traise ValueError("Debes proporcionar horario o horario_id")\n\n\t\tif isinstance(horario, str):\n\t\t\thorario_id = horario\n\t\t\thorario = None\n\n\t\tif horario is None:\n\t\t\thorario = self.obtener_horario_desde_id_cache(horario_id)\n\n\t\tif horario_id is None:\n\t\t\thorario_id = horario_id_desde_horario(horario)\n\n\t\treturn self.score_horario_regular(\n\t\t\teco=eco_texto,\n\t\t\thorario_id=horario_id,\n\t\t\thorario=horario,\n\t\t)\n'
Path("ih_core.py").write_text(IH_CORE_SOURCE, encoding="utf-8")

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from ih_core import *

print("ih_core.py listo para importar en Colab")


In [ ]:
from pathlib import Path

ARCHIVOS_REQUERIDOS = [
    "df_hist.json",
    "ecos_vigentes_con_horario_regular.json",
    "ecos_vigentes_con_horario_irregular.json",
]

ARCHIVOS_OPCIONALES = [
    "ecos_irregulares_inferidos.json",
    "eco-nombre.json",
]

def archivos_faltantes(nombres):
    return [nombre for nombre in nombres if not (BASE_DIR / nombre).exists()]

faltantes = archivos_faltantes(ARCHIVOS_REQUERIDOS)

if faltantes and EN_COLAB:
    from google.colab import files
    print("Sube los JSON requeridos y, si los tienes, tambi?n los opcionales:")
    print("Requeridos:", ", ".join(ARCHIVOS_REQUERIDOS))
    print("Opcionales:", ", ".join(ARCHIVOS_OPCIONALES))
    uploaded = files.upload()
    for nombre, contenido in uploaded.items():
        destino = BASE_DIR / nombre
        destino.write_bytes(contenido)
        print(f"Guardado: {destino.name}")

faltantes = archivos_faltantes(ARCHIVOS_REQUERIDOS)
if faltantes:
    raise FileNotFoundError(
        "Faltan archivos requeridos en "
        f"{BASE_DIR}: {', '.join(faltantes)}"
    )

print("Archivos requeridos listos")


In [ ]:
import sys
import os
from pathlib import Path
import joblib

from ih_core import *


In [ ]:
rutas_archivos = {
	"historico": Path("df_hist.json"),
	"regulares": Path("ecos_vigentes_con_horario_regular.json"),
	"irregulares": Path("ecos_vigentes_con_horario_irregular.json"),
	"irregulares_inferidos": Path("ecos_irregulares_inferidos.json"),
	"nombres": Path("eco-nombre.json"),
}
df_hist = pd.DataFrame(cargar_json(rutas_archivos["historico"]))
ecos_regulares = cargar_json(rutas_archivos["regulares"])
ecos_irregulares = cargar_json(rutas_archivos["irregulares"])
ecos_irregulares_inferidos = cargar_json(rutas_archivos["irregulares_inferidos"]) if rutas_archivos["irregulares_inferidos"].exists() else {}
eco_nombres = cargar_json(rutas_archivos["nombres"]) if rutas_archivos["nombres"].exists() else {}



In [ ]:
admin_windows = {
	"L": [
		{"inicio": "10:00", "fin": "14:00"},
		{"inicio": "16:00", "fin": "18:00"},
	],
	"M": [
		{"inicio": "10:00", "fin": "14:00"},
		{"inicio": "16:00", "fin": "18:00"},
	],
	"Mi": [
		{"inicio": "10:00", "fin": "14:00"},
		{"inicio": "16:00", "fin": "18:00"},
	],
	"J": [
		{"inicio": "10:00", "fin": "14:00"},
		{"inicio": "16:00", "fin": "18:00"},
	],
	"V": [
		{"inicio": "10:00", "fin": "14:00"},
		{"inicio": "16:00", "fin": "18:00"},
	],
}
hiperparametros = {
	"lambda_in": 1.0,
	"lambda_out": 0.9,
	"lambda_admin_pos": 1.0,
	"gamma_contrato": 0.9,
	"gamma_admin": 0.15,
	"alpha_out": 1.0,
	"half_life_trimestres": 8.0,
}
modelo_ih_regular = ModeloPrediccionIHRegular(**hiperparametros).fit(
	df_hist=df_hist,
	ecos_regulares=ecos_regulares,
	ecos_irregulares=ecos_irregulares,
	admin_windows=admin_windows,
)
modelo_ih_irregular = ModeloPrediccionIHIrregular(**hiperparametros).fit(
	df_hist=df_hist,
	ecos_irregulares_inferidos=ecos_irregulares_inferidos,
	admin_windows=admin_windows,
)
fixed_window_regulares = modelo_ih_regular.fixed_window_regulares
prior_global_h = modelo_ih_regular.prior_global_h
count_out_exact = modelo_ih_regular.count_out_exact
total_out = modelo_ih_regular.total_out
resumen_entrenamiento = {
	"tri_num_actual": modelo_ih_regular.tri_num_actual,
	"ecos_regulares": len(modelo_ih_regular.ecos_regulares),
	"ecos_irregulares_cargados": len(modelo_ih_regular.ecos_irregulares),
	"ecos_irregulares_inferidos": len(modelo_ih_irregular.ecos_irregulares_inferidos),
	"horarios_globales_observados": len(prior_global_h),
	"ecos_con_historia_fuera_de_bloque": sum(
		1 for total_eco in total_out.values() if total_eco > 0
	),
}
resumen_entrenamiento



In [ ]:
eco_prediccion = "4377"
horario = "14:30-16:00"
horario_id_prediccion = "L:"+horario+"|Mi:"+horario+"|V:"+horario

prediccion = modelo_ih_regular.predecir(
	eco=eco_prediccion,
	horario_id=horario_id_prediccion,
)

prediccion


In [ ]:
import math
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "modelo_ih_regular" not in globals():
	raise ValueError("No existe modelo_ih_regular en memoria. Entrena o carga el modelo antes de correr esta celda.")

objetivos_ih = [
	{
		"eco": "4377",
		"horario_id": "L:14:30-16:00|Mi:14:30-16:00|V:14:30-16:00",
		"target": 0.75,
		"peso": 1.0,
	},
	{
		"eco": "4377",
		"horario_id": "L:16:00-17:30|Mi:16:00-17:30|V:16:00-17:30",
		"target": 0.75,
		"peso": 1.0,
	},
	{
		"eco": "4377",
		"horario_id": "L:13:00-14:30|Mi:13:00-14:30|V:13:00-14:30",
		"target": 0.75,
		"peso": 1.0,
	},
	{
		"eco": "28650",
		"horario_id": "L:16:30-17:30|Mi:16:30-17:30|V:16:30-17:30",
		"target": 0.90,
		"peso": 1.0,
	},
	{
		"eco": "28650",
		"horario_id": "L:17:30-19:00|Mi:17:30-19:00|V:17:30-19:00",
		"target": 0.75,
		"peso": 1.0,
	},
]

parametros_base_afinidad_ih = {
	"alpha_out": float(modelo_ih_regular.alpha_out),
	"rho_exact": float(modelo_ih_regular.rho_exact),
	"w_overlap": float(modelo_ih_regular.w_overlap),
	"w_duration": float(modelo_ih_regular.w_duration),
	"w_days": float(modelo_ih_regular.w_days),
	"w_time": float(modelo_ih_regular.w_time),
}

parametros_base_score_ih = {
	"lambda_in": float(modelo_ih_regular.lambda_in),
	"lambda_out": float(modelo_ih_regular.lambda_out),
	"lambda_admin_pos": float(modelo_ih_regular.lambda_admin_pos),
	"gamma_contrato": float(modelo_ih_regular.gamma_contrato),
	"gamma_admin": float(modelo_ih_regular.gamma_admin),
}

rangos_afinidad_ih = {
	"alpha_out": (0.25, 4.0),
	"rho_exact": (0.05, 0.85),
	"w_overlap": (0.10, 0.90),
	"w_duration": (0.05, 0.60),
	"w_days": (0.05, 0.60),
	"w_time": (0.05, 0.60),
}

rangos_score_ih = {
	"lambda_in": (0.40, 1.20),
	"lambda_out": (0.50, 2.50),
	"lambda_admin_pos": (0.00, 0.80),
	"gamma_contrato": (0.05, 0.80),
	"gamma_admin": (0.00, 0.50),
}

peso_regularizacion_afinidad_ih = 0.05
peso_regularizacion_score_ih = 0.05

puntos_grid_inicial_afinidad = 3
puntos_grid_refinado_afinidad = 3
puntos_grid_inicial_score = 3
puntos_grid_refinado_score = 3
numero_refinamientos = 2
factor_contraccion = 0.40

def normalizar_pesos_kernel(raw_weights):
	vector = np.array(
		[
			max(0.0, float(raw_weights["w_overlap"])),
			max(0.0, float(raw_weights["w_duration"])),
			max(0.0, float(raw_weights["w_days"])),
			max(0.0, float(raw_weights["w_time"])),
		],
		dtype=float,
	)
	suma = float(vector.sum())
	if suma <= 1e-12:
		vector = np.array([0.25, 0.25, 0.25, 0.25], dtype=float)
	else:
		vector = vector / suma
	return {
		"w_overlap": float(vector[0]),
		"w_duration": float(vector[1]),
		"w_days": float(vector[2]),
		"w_time": float(vector[3]),
	}

def aplicar_parametros_modelo(modelo, params_afinidad, params_score):
	for nombre, valor in params_afinidad.items():
		setattr(modelo, nombre, float(valor))
	for nombre, valor in params_score.items():
		setattr(modelo, nombre, float(valor))

def construir_grid_desde_rangos(rangos, puntos_por_parametro):
	return {
		parametro: np.linspace(limite_inferior, limite_superior, puntos_por_parametro)
		for parametro, (limite_inferior, limite_superior) in rangos.items()
	}

def contraer_rangos(rangos_base, mejor_params, factor):
	rangos_contraidos = {}
	for parametro, (limite_inferior, limite_superior) in rangos_base.items():
		centro = float(mejor_params[parametro])
		span_total = limite_superior - limite_inferior
		nuevo_span = span_total * factor
		nuevo_inferior = max(limite_inferior, centro - nuevo_span / 2.0)
		nuevo_superior = min(limite_superior, centro + nuevo_span / 2.0)
		if math.isclose(nuevo_inferior, nuevo_superior):
			nuevo_inferior = centro
			nuevo_superior = centro
		rangos_contraidos[parametro] = (nuevo_inferior, nuevo_superior)
	return rangos_contraidos

def calcular_regularizacion_normalizada(params, params_base, rangos, peso_regularizacion):
	acumulado = 0.0
	for parametro, valor in params.items():
		limite_inferior, limite_superior = rangos[parametro]
		span = limite_superior - limite_inferior
		if span <= 1e-12:
			continue
		acumulado += ((float(valor) - float(params_base[parametro])) / span) ** 2
	return peso_regularizacion * acumulado

def precomputar_estadisticas_objetivos(modelo, objetivos):
	rows = []

	for objetivo in objetivos:
		eco = str(objetivo["eco"])
		horario_id = str(objetivo["horario_id"])
		horario_objetivo = modelo.obtener_horario_desde_id_cache(horario_id)

		afinidad_contrato = float(modelo.cobertura_contrato_regular(eco, horario_objetivo))
		preferencia_administrativa = float(modelo.preferencia_admin(horario_objetivo))
		prior_global_h = float(modelo.prior_global_h.get(horario_id, 0.0))
		conteo_exacto = float(modelo.count_out_exact.get(eco, {}).get(horario_id, 0.0))
		total_out = float(modelo.total_out.get(eco, 0.0))

		registros = modelo.registros_fuera_regular.get(eco, [])

		masa_overlap = 0.0
		masa_duration = 0.0
		masa_days = 0.0
		masa_time = 0.0

		for registro in registros:
			peso = float(registro["peso"])
			horario_historico = registro["horario"]

			masa_overlap += peso * float(similitud_horaria_por_traslape(horario_objetivo, horario_historico))
			masa_duration += peso * float(similitud_duracion_horaria(horario_objetivo, horario_historico))
			masa_days += peso * float(similitud_patron_dias(horario_objetivo, horario_historico))
			masa_time += peso * float(similitud_tiempo_circular(horario_objetivo, horario_historico))

		if total_out > 0.0:
			componente_overlap = masa_overlap / total_out
			componente_duration = masa_duration / total_out
			componente_days = masa_days / total_out
			componente_time = masa_time / total_out
		else:
			componente_overlap = 0.0
			componente_duration = 0.0
			componente_days = 0.0
			componente_time = 0.0

		rows.append(
			{
				"eco": eco,
				"horario_id": horario_id,
				"target": float(objetivo["target"]),
				"peso": float(objetivo.get("peso", 1.0)),
				"afinidad_contrato": afinidad_contrato,
				"preferencia_administrativa": preferencia_administrativa,
				"prior_global_h": prior_global_h,
				"conteo_exacto_fuera": conteo_exacto,
				"total_out": total_out,
				"componente_overlap": componente_overlap,
				"componente_duration": componente_duration,
				"componente_days": componente_days,
				"componente_time": componente_time,
			}
		)

	return pd.DataFrame(rows)

def evaluar_con_parametros(stats_df, params_afinidad, params_score):
	pesos_kernel = normalizar_pesos_kernel(params_afinidad)

	alpha_out = float(params_afinidad["alpha_out"])
	rho_exact = max(0.0, min(1.0, float(params_afinidad["rho_exact"])))

	conteo_exacto = stats_df["conteo_exacto_fuera"].to_numpy(dtype=float)
	total_out = stats_df["total_out"].to_numpy(dtype=float)
	prior_global_h = stats_df["prior_global_h"].to_numpy(dtype=float)

	afinidad_exacta = (conteo_exacto + alpha_out * prior_global_h) / (total_out + alpha_out)

	afinidad_overlap = (
		pesos_kernel["w_overlap"] * stats_df["componente_overlap"].to_numpy(dtype=float)
		+ pesos_kernel["w_duration"] * stats_df["componente_duration"].to_numpy(dtype=float)
		+ pesos_kernel["w_days"] * stats_df["componente_days"].to_numpy(dtype=float)
		+ pesos_kernel["w_time"] * stats_df["componente_time"].to_numpy(dtype=float)
	)

	afinidad_total = rho_exact * afinidad_exacta + (1.0 - rho_exact) * afinidad_overlap

	afinidad_contrato = stats_df["afinidad_contrato"].to_numpy(dtype=float)
	preferencia_administrativa = stats_df["preferencia_administrativa"].to_numpy(dtype=float)
	target = stats_df["target"].to_numpy(dtype=float)
	peso = stats_df["peso"].to_numpy(dtype=float)

	lambda_in = float(params_score["lambda_in"])
	lambda_out = float(params_score["lambda_out"])
	lambda_admin_pos = float(params_score["lambda_admin_pos"])
	gamma_contrato = float(params_score["gamma_contrato"])
	gamma_admin = float(params_score["gamma_admin"])

	bonus_contrato = lambda_in * afinidad_contrato
	bonus_fuera_justificado = lambda_out * (1.0 - afinidad_contrato) * afinidad_total
	bonus_admin = lambda_admin_pos * preferencia_administrativa

	penalizacion_fuera_no_justificada = gamma_contrato * (1.0 - afinidad_contrato) * (1.0 - afinidad_total)
	penalizacion_admin = gamma_admin * (1.0 - preferencia_administrativa)

	score_lineal = (
		bonus_contrato
		+ bonus_fuera_justificado
		+ bonus_admin
		- penalizacion_fuera_no_justificada
		- penalizacion_admin
	)

	pred = np.clip(score_lineal, 0.0, 1.0)
	error = pred - target
	abs_error = np.abs(error)
	weighted_sq_error = peso * (error ** 2)

	df_eval = stats_df[["eco", "horario_id", "target", "peso"]].copy()
	df_eval["pred"] = pred
	df_eval["error"] = error
	df_eval["abs_error"] = abs_error
	df_eval["weighted_sq_error"] = weighted_sq_error
	df_eval["afinidad_contrato"] = afinidad_contrato
	df_eval["afinidad_fuera_historica_exacta"] = afinidad_exacta
	df_eval["afinidad_fuera_historica_overlap"] = afinidad_overlap
	df_eval["afinidad_fuera_historica"] = afinidad_total
	df_eval["rho_exact"] = rho_exact
	df_eval["prior_global_h"] = prior_global_h
	df_eval["preferencia_administrativa"] = preferencia_administrativa
	df_eval["bonus_contrato"] = bonus_contrato
	df_eval["bonus_fuera_justificado"] = bonus_fuera_justificado
	df_eval["bonus_admin"] = bonus_admin
	df_eval["penalizacion_fuera_no_justificada"] = penalizacion_fuera_no_justificada
	df_eval["penalizacion_admin"] = penalizacion_admin
	df_eval["score_lineal"] = score_lineal
	df_eval["w_overlap_norm"] = pesos_kernel["w_overlap"]
	df_eval["w_duration_norm"] = pesos_kernel["w_duration"]
	df_eval["w_days_norm"] = pesos_kernel["w_days"]
	df_eval["w_time_norm"] = pesos_kernel["w_time"]

	return float(weighted_sq_error.sum()), float(abs_error.sum()), df_eval

def evaluar_grid_afinidad(stats_df, grid_afinidad, params_score_fijos, params_base_afinidad, rangos_afinidad, peso_regularizacion, historial, etiqueta):
	claves_grid = list(grid_afinidad.keys())
	valores_grid = [grid_afinidad[clave] for clave in claves_grid]

	best_loss_total = float("inf")
	best_loss_datos = float("inf")
	best_abs = float("inf")
	best_reg = float("inf")
	best_params = None
	best_df_eval = None
	total_evaluadas = 0

	for combinacion in itertools.product(*valores_grid):
		total_evaluadas += 1
		params_afinidad = {
			clave: float(valor)
			for clave, valor in zip(claves_grid, combinacion)
		}

		loss_datos, abs_loss, df_eval = evaluar_con_parametros(
			stats_df=stats_df,
			params_afinidad=params_afinidad,
			params_score=params_score_fijos,
		)

		reg = calcular_regularizacion_normalizada(
			params=params_afinidad,
			params_base=params_base_afinidad,
			rangos=rangos_afinidad,
			peso_regularizacion=peso_regularizacion,
		)

		loss_total = loss_datos + reg

		if (loss_total < best_loss_total) or (math.isclose(loss_total, best_loss_total) and abs_loss < best_abs):
			best_loss_total = loss_total
			best_loss_datos = loss_datos
			best_abs = abs_loss
			best_reg = reg
			best_params = dict(params_afinidad)
			best_df_eval = df_eval.copy()

	historial.append(
		{
			"bloque": etiqueta,
			"loss_total": best_loss_total,
			"loss_datos": best_loss_datos,
			"regularizacion": best_reg,
			"best_params": dict(best_params),
		}
	)

	return {
		"best_loss_total": best_loss_total,
		"best_loss_datos": best_loss_datos,
		"best_abs": best_abs,
		"best_reg": best_reg,
		"best_params": best_params,
		"best_df_eval": best_df_eval,
		"total_evaluadas": total_evaluadas,
	}

def evaluar_grid_score(stats_df, params_afinidad_fijos, grid_score, params_base_score, rangos_score, peso_regularizacion, historial, etiqueta):
	claves_grid = list(grid_score.keys())
	valores_grid = [grid_score[clave] for clave in claves_grid]

	best_loss_total = float("inf")
	best_loss_datos = float("inf")
	best_abs = float("inf")
	best_reg = float("inf")
	best_params = None
	best_df_eval = None
	total_evaluadas = 0

	for combinacion in itertools.product(*valores_grid):
		total_evaluadas += 1
		params_score = {
			clave: float(valor)
			for clave, valor in zip(claves_grid, combinacion)
		}

		loss_datos, abs_loss, df_eval = evaluar_con_parametros(
			stats_df=stats_df,
			params_afinidad=params_afinidad_fijos,
			params_score=params_score,
		)

		reg = calcular_regularizacion_normalizada(
			params=params_score,
			params_base=params_base_score,
			rangos=rangos_score,
			peso_regularizacion=peso_regularizacion,
		)

		loss_total = loss_datos + reg

		if (loss_total < best_loss_total) or (math.isclose(loss_total, best_loss_total) and abs_loss < best_abs):
			best_loss_total = loss_total
			best_loss_datos = loss_datos
			best_abs = abs_loss
			best_reg = reg
			best_params = dict(params_score)
			best_df_eval = df_eval.copy()

	historial.append(
		{
			"bloque": etiqueta,
			"loss_total": best_loss_total,
			"loss_datos": best_loss_datos,
			"regularizacion": best_reg,
			"best_params": dict(best_params),
		}
	)

	return {
		"best_loss_total": best_loss_total,
		"best_loss_datos": best_loss_datos,
		"best_abs": best_abs,
		"best_reg": best_reg,
		"best_params": best_params,
		"best_df_eval": best_df_eval,
		"total_evaluadas": total_evaluadas,
	}

stats_objetivos_ih = precomputar_estadisticas_objetivos(modelo_ih_regular, objetivos_ih)

params_afinidad_actuales = dict(parametros_base_afinidad_ih)
params_score_actuales = dict(parametros_base_score_ih)

loss_baseline_datos_ih, abs_baseline_ih, df_baseline_ih = evaluar_con_parametros(
	stats_df=stats_objetivos_ih,
	params_afinidad=params_afinidad_actuales,
	params_score=params_score_actuales,
)
loss_baseline_total_ih = loss_baseline_datos_ih

historial_busqueda_ih = []
total_combinaciones_ih = 0

rangos_afinidad_actuales = dict(rangos_afinidad_ih)
rangos_score_actuales = dict(rangos_score_ih)

resultado_afinidad = evaluar_grid_afinidad(
	stats_df=stats_objetivos_ih,
	grid_afinidad=construir_grid_desde_rangos(rangos_afinidad_actuales, puntos_grid_inicial_afinidad),
	params_score_fijos=params_score_actuales,
	params_base_afinidad=parametros_base_afinidad_ih,
	rangos_afinidad=rangos_afinidad_ih,
	peso_regularizacion=peso_regularizacion_afinidad_ih,
	historial=historial_busqueda_ih,
	etiqueta="afinidad_0",
)
params_afinidad_actuales = dict(resultado_afinidad["best_params"])
best_loss_total_ih = resultado_afinidad["best_loss_total"]
best_loss_datos_ih = resultado_afinidad["best_loss_datos"]
best_abs_ih = resultado_afinidad["best_abs"]
best_reg_ih = resultado_afinidad["best_reg"]
best_df_eval_ih = resultado_afinidad["best_df_eval"].copy()
total_combinaciones_ih += resultado_afinidad["total_evaluadas"]

resultado_score = evaluar_grid_score(
	stats_df=stats_objetivos_ih,
	params_afinidad_fijos=params_afinidad_actuales,
	grid_score=construir_grid_desde_rangos(rangos_score_actuales, puntos_grid_inicial_score),
	params_base_score=parametros_base_score_ih,
	rangos_score=rangos_score_ih,
	peso_regularizacion=peso_regularizacion_score_ih,
	historial=historial_busqueda_ih,
	etiqueta="score_0",
)
params_score_actuales = dict(resultado_score["best_params"])
best_loss_total_ih = resultado_score["best_loss_total"]
best_loss_datos_ih = resultado_score["best_loss_datos"]
best_abs_ih = resultado_score["best_abs"]
best_reg_ih = resultado_score["best_reg"]
best_df_eval_ih = resultado_score["best_df_eval"].copy()
total_combinaciones_ih += resultado_score["total_evaluadas"]

for indice_refinamiento in range(1, numero_refinamientos + 1):
	rangos_afinidad_actuales = contraer_rangos(rangos_afinidad_actuales, params_afinidad_actuales, factor_contraccion)

	resultado_afinidad = evaluar_grid_afinidad(
		stats_df=stats_objetivos_ih,
		grid_afinidad=construir_grid_desde_rangos(rangos_afinidad_actuales, puntos_grid_refinado_afinidad),
		params_score_fijos=params_score_actuales,
		params_base_afinidad=parametros_base_afinidad_ih,
		rangos_afinidad=rangos_afinidad_ih,
		peso_regularizacion=peso_regularizacion_afinidad_ih,
		historial=historial_busqueda_ih,
		etiqueta=f"afinidad_{indice_refinamiento}",
	)
	params_afinidad_actuales = dict(resultado_afinidad["best_params"])
	total_combinaciones_ih += resultado_afinidad["total_evaluadas"]

	rangos_score_actuales = contraer_rangos(rangos_score_actuales, params_score_actuales, factor_contraccion)

	resultado_score = evaluar_grid_score(
		stats_df=stats_objetivos_ih,
		params_afinidad_fijos=params_afinidad_actuales,
		grid_score=construir_grid_desde_rangos(rangos_score_actuales, puntos_grid_refinado_score),
		params_base_score=parametros_base_score_ih,
		rangos_score=rangos_score_ih,
		peso_regularizacion=peso_regularizacion_score_ih,
		historial=historial_busqueda_ih,
		etiqueta=f"score_{indice_refinamiento}",
	)
	params_score_actuales = dict(resultado_score["best_params"])
	total_combinaciones_ih += resultado_score["total_evaluadas"]

	if (resultado_score["best_loss_total"] < best_loss_total_ih) or (
		math.isclose(resultado_score["best_loss_total"], best_loss_total_ih)
		and resultado_score["best_abs"] < best_abs_ih
	):
		best_loss_total_ih = resultado_score["best_loss_total"]
		best_loss_datos_ih = resultado_score["best_loss_datos"]
		best_abs_ih = resultado_score["best_abs"]
		best_reg_ih = resultado_score["best_reg"]
		best_df_eval_ih = resultado_score["best_df_eval"].copy()

params_afinidad_finales = dict(params_afinidad_actuales)
params_afinidad_finales.update(normalizar_pesos_kernel(params_afinidad_finales))
params_score_finales = dict(params_score_actuales)

aplicar_parametros_modelo(
	modelo=modelo_ih_regular,
	params_afinidad=params_afinidad_finales,
	params_score=params_score_finales,
)

df_parametros_base_ih = pd.DataFrame(
	[
		{"grupo": "afinidad", "parametro": clave, "valor": float(valor)}
		for clave, valor in parametros_base_afinidad_ih.items()
	]
	+ [
		{"grupo": "score", "parametro": clave, "valor": float(valor)}
		for clave, valor in parametros_base_score_ih.items()
	]
).sort_values(["grupo", "parametro"]).reset_index(drop=True)

df_parametros_optimos_ih = pd.DataFrame(
	[
		{"grupo": "afinidad", "parametro": clave, "valor": float(valor)}
		for clave, valor in params_afinidad_finales.items()
	]
	+ [
		{"grupo": "score", "parametro": clave, "valor": float(valor)}
		for clave, valor in params_score_finales.items()
	]
).sort_values(["grupo", "parametro"]).reset_index(drop=True)

df_historial_ih = pd.DataFrame(
	[
		{
			"bloque": fila["bloque"],
			"loss_total": float(fila["loss_total"]),
			"loss_datos": float(fila["loss_datos"]),
			"regularizacion": float(fila["regularizacion"]),
		}
		for fila in historial_busqueda_ih
	]
)

print(f"Combinaciones evaluadas: {total_combinaciones_ih:,}")

print("Parámetros base")
display(df_parametros_base_ih)

print(f"Loss baseline datos: {loss_baseline_datos_ih:.10f}")
print(f"Loss baseline total: {loss_baseline_total_ih:.10f}")
print(f"Abs baseline: {abs_baseline_ih:.10f}")

print("Resultados baseline")
display(df_baseline_ih)

print("Mejores hiperparámetros")
display(df_parametros_optimos_ih)

print(f"Loss óptimo datos: {best_loss_datos_ih:.10f}")
print(f"Regularización óptima: {best_reg_ih:.10f}")
print(f"Loss óptimo total: {best_loss_total_ih:.10f}")
print(f"Abs óptimo: {best_abs_ih:.10f}")

print("Resultados óptimos")
display(best_df_eval_ih)

plt.figure(figsize=(12, 6))
plt.plot(
	range(1, len(df_historial_ih) + 1),
	df_historial_ih["loss_total"],
	linewidth=1.0,
	label="Loss total",
)
plt.plot(
	range(1, len(df_historial_ih) + 1),
	df_historial_ih["loss_datos"],
	linewidth=0.9,
	alpha=0.75,
	label="Loss datos",
)
plt.scatter(
	int(df_historial_ih["loss_total"].idxmin()) + 1,
	float(df_historial_ih["loss_total"].min()),
	s=90,
	edgecolors="black",
	zorder=5,
	label="Mejor bloque",
)

plt.xticks(range(1, len(df_historial_ih) + 1), df_historial_ih["bloque"], rotation=45)
plt.title("Grid search por bloques para el nuevo modelo de afinidad")
plt.xlabel("Bloque de búsqueda")
plt.ylabel("Loss")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import joblib

if "modelo_ih_regular" not in globals():
	raise ValueError("No existe modelo_ih_regular en memoria. Ejecuta primero el entrenamiento y el grid search.")

artefacto_afinidad_ih = {
	"modelo": modelo_ih_regular,
	"params_afinidad": params_afinidad_finales if "params_afinidad_finales" in globals() else None,
	"params_score": params_score_finales if "params_score_finales" in globals() else None,
}

joblib.dump(artefacto_afinidad_ih, "afinidad_ih.joblib")

print("Artefacto guardado en afinidad_ih.joblib")

In [ ]:
artefacto = joblib.load("afinidad_ih.joblib")

if isinstance(artefacto, dict) and "modelo" in artefacto:
	modelo_ih_regular = artefacto["modelo"]
	params_afinidad_finales = artefacto.get("params_afinidad")
	params_score_finales = artefacto.get("params_score")
else:
	modelo_ih_regular = artefacto

In [ ]:
eco_prediccion = "4377"
horario = "14:30-16:00"
horario_id_prediccion = "L:"+horario+"|Mi:"+horario+"|V:"+horario

prediccion = modelo_ih_regular.predecir(
	eco=eco_prediccion,
	horario_id=horario_id_prediccion,
)

prediccion

In [ ]:
import joblib
artefacto_final = {
    'regular': modelo_ih_regular,
    'irregular': modelo_ih_irregular
}
joblib.dump(artefacto_final, 'ih_model.joblib')
print('Modelo guardado exitosamente en ih_model.joblib')



In [ ]:
# Opcional: descargar artefactos generados desde Colab
if EN_COLAB:
    from google.colab import files
    for nombre in ["afinidad_ih.joblib", "ih_model.joblib"]:
        ruta = Path(nombre)
        if ruta.exists():
            files.download(str(ruta))
